# RealPage SFTP Download
Descarga los CSV de las tablas seleccionadas desde el servidor SFTP de RealPage.
Requiere: `pip install paramiko`

In [1]:
# %pip install paramiko

In [2]:
import os
import re
import stat
import pathlib
import datetime as dt
import zoneinfo
import paramiko

# -------------------- SETTINGS --------------------
HOST, PORT = "bimft.realpage.com", 22
USER       = "EXT-HERITAGEHILLPROPMGMT_BI"
PWD        = "BJEEod^ykvA9Rm"
REMOTE_DIR = "/BI_Download"

# Carpeta local donde se guardan los CSV
LOCAL_DIR = pathlib.Path(os.path.dirname(os.path.abspath("__file__")))
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Tablas a descargar
TARGET_TABLES = [
    "dimaccountgrouphierarchy__0001",
    "dimclassificationlist__0001",
    "dimleadmanagement__0001",
    "dimleadmetrics__0001",
    "dimleaseattributes__0001",
    "dimresidentactivitylog__0001",
    "dimleaseexpiration__0001",
    "dimmakereadyrequest__0001",
    "dimproperty__0001",
    "dimresidentmember__0001",
    "dimscreeningapplicant__0001",
    "dimservicerequest__0001",
    "factoperationalkpi__0001",
    "factaccountgrouphierarchydetail__0001",
    "factglsummary__0001",
    "factpropertystatisticsasofdate__0001",
    "factscreeningapplicant__0001",
]

def _norm(s: str) -> str:
    """Normaliza nombre para comparación case-insensitive sin caracteres especiales."""
    return re.sub(r"[^a-z0-9]+", "", (s or "").lower())

TARGET_SET = {_norm(t) for t in TARGET_TABLES}

def is_target_file(filename: str) -> bool:
    """Devuelve True si el archivo corresponde a una de las tablas objetivo."""
    stem = re.sub(r"\.csv$", "", filename, flags=re.I)
    # Quitar prefijo de fecha/timestamp y 'dbo__' si existe
    clean = re.sub(r"^.*?dbo__", "", stem, flags=re.I)
    if not clean:
        clean = stem
    return _norm(clean) in TARGET_SET

def clean_filename(filename: str) -> str:
    """Retorna nombre limpio: solo la parte después de dbo__ (o el nombre original)."""
    stem = re.sub(r"\.csv$", "", filename, flags=re.I)
    m = re.search(r"dbo__(.+)", stem, flags=re.I)
    return (m.group(1) if m else stem) + ".csv"

print(f"Local dir : {LOCAL_DIR}")
print(f"Tablas objetivo: {len(TARGET_TABLES)}")

Local dir : c:\Users\MarioSoler\OneDrive - Heritage Hill Property Management, LLC\MCP_Fabric\BIX_Examples
Tablas objetivo: 17


In [3]:
# -------------------- DESCARGA SFTP --------------------
downloaded = []
skipped    = []

with paramiko.Transport((HOST, PORT)) as tr:
    tr.connect(username=USER, password=PWD)
    print("Conectado al servidor SFTP")

    with paramiko.SFTPClient.from_transport(tr) as sftp:

        # Listar directorio remoto
        try:
            entries = sftp.listdir_attr(REMOTE_DIR)
        except FileNotFoundError:
            entries = sftp.listdir_attr(REMOTE_DIR.lstrip("/"))

        print(f"Archivos en {REMOTE_DIR}: {len(entries)}")

        for entry in entries:
            if not stat.S_ISREG(entry.st_mode):
                continue  # saltar carpetas
            if not entry.filename.lower().endswith(".csv"):
                continue  # solo CSV

            if not is_target_file(entry.filename):
                skipped.append(entry.filename)
                continue

            dest_name  = clean_filename(entry.filename)
            local_path = LOCAL_DIR / dest_name
            remote_path = f"{REMOTE_DIR}/{entry.filename}"

            print(f"  Descargando: {entry.filename}  ->  {dest_name}")
            sftp.get(remote_path, str(local_path))
            downloaded.append(dest_name)

print(f"\nDescargados : {len(downloaded)}")
print(f"Omitidos    : {len(skipped)}")
print("\nArchivos descargados:")
for f in downloaded:
    print(f"  {f}")

Conectado al servidor SFTP
Archivos en /BI_Download: 4798
  Descargando: RA3712076__20260606__dbo__DimAccountGroupHierarchy__0001.csv  ->  DimAccountGroupHierarchy__0001.csv
  Descargando: RA3712076__20260606__dbo__DIMClassificationList__0001.csv  ->  DIMClassificationList__0001.csv
  Descargando: RA3712076__20260606__dbo__DimLeadManagement__0001.csv  ->  DimLeadManagement__0001.csv
  Descargando: RA3712076__20260606__dbo__DimLeadMetrics__0001.csv  ->  DimLeadMetrics__0001.csv
  Descargando: RA3712076__20260606__dbo__DimLeaseAttributes__0001.csv  ->  DimLeaseAttributes__0001.csv
  Descargando: RA3712076__20260606__dbo__DimLeaseExpiration__0001.csv  ->  DimLeaseExpiration__0001.csv
  Descargando: RA3712076__20260606__dbo__DimMakeReadyRequest__0001.csv  ->  DimMakeReadyRequest__0001.csv
  Descargando: RA3712076__20260606__dbo__DimProperty__0001.csv  ->  DimProperty__0001.csv
  Descargando: RA3712076__20260606__dbo__DimResidentActivityLog__0001.csv  ->  DimResidentActivityLog__0001.csv
  